# Предсказание цен на жилье в городе Эймс, штат Айова, США

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [10]:
from config import config

In [11]:
train_df = pd.read_csv(config.train_data_path)
test_df = pd.read_csv(config.test_data_path)

In [12]:
train_df.head(5)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [13]:
print("train_df shape:", train_df.shape)
print("test_df shape:", test_df.shape)

train_df shape: (1460, 81)
test_df shape: (1459, 80)


In [14]:
from sklearn.model_selection import train_test_split

X = train_df.drop(columns=['Id', 'SalePrice'])
y = train_df['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [15]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (1022, 79)
y_train shape: (1022,)
X_test shape: (438, 79)
y_test shape: (438,)


Отберем все числовые и категориальные признаки и заполним пропуски

In [16]:
num_cols = train_df.select_dtypes(include=['int64', 'float64']).columns.to_list()
y = train_df['SalePrice']
cols_to_drop = ['Id', 'SalePrice', 'MSSubClass']
num_cols = [col for col in num_cols if col not in cols_to_drop]
cat_cols = train_df.select_dtypes(include=['str']).columns.to_list()
cat_cols.append('MSSubClass')

In [17]:
print(f"{len(num_cols)} числовых признаков")
print(f"{len(cat_cols)} категориальных признаков")

35 числовых признаков
44 категориальных признаков


## Просмотр данных

In [18]:
num_train_df = train_df[num_cols]
cat_train_df = train_df[cat_cols]

Гистограммы для тренировочного датасета:

In [19]:
# num_train_df.hist(figsize=(20, 15), bins=50)
# plt.tight_layout()
# plt.show()

Матрица корреляций для числовых признаков:

In [20]:
# corr = num_train_df.corr()

# plt.figure(figsize=(14, 12))
# sns.heatmap(corr, annot=False, fmt='.2f', cmap='coolwarm', square=True)
# plt.title('Корреляционная матрица')
# plt.show()

In [21]:
# from src.utils import visualize_categorical_features

# visualize_categorical_features(train_df, cat_cols)

## Обработка данных

In [22]:
from src.utils import CategoricalTransformer, NumericalTransformer

num_transformer = NumericalTransformer(num_cols=num_cols,   
                                       impute_strategy='median', 
                                       scaling=False,
                                       scaler_type='robust')                                       

cat_transformer = CategoricalTransformer(cat_cols=cat_cols,                                         
                                         encoding=True,
                                         fill_suffix="No")

In [23]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

In [24]:
from src.utils.utils import check_missing_values

preprocessor.fit(X_train)

all_features = preprocessor.get_feature_names_out()

X_train_preprocessed = pd.DataFrame(preprocessor.transform(X_train), 
                                    columns=list(all_features))
if sum(check_missing_values(X_train_preprocessed)) > 0:
    print("В данных есть пропущенные значения:")   
else:
    print("Пропущенных значений нет") 

check_missing_values(X_train_preprocessed)
print(X_train_preprocessed.shape)

Пропущенных значений нет
(1022, 268)


#### Испытания RandomForestRegressor

In [25]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA

rf_test_model = RandomForestRegressor(
    n_estimators=200,           
    max_depth=15,               
    min_samples_split=10,       
    min_samples_leaf=5,         
    max_features='sqrt',        
    bootstrap=True,             
    random_state=42,
    n_jobs=-1                   
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),    
    ('model', rf_test_model)
])

pipeline.fit(X_train, y_train)

train_predictions = pipeline.predict(X_train)
test_predictions = pipeline.predict(X_test)

c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [10, 14, 15, 29] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [26]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from src.utils import show_metrics

print("=== МЕТРИКИ НА ТРЕНИРОВОЧНЫХ ДАННЫХ ===")
show_metrics(train_predictions, y_train)

print("----------------------------------")

print("=== МЕТРИКИ НА ТЕСТОВЫХ ДАННЫХ ===")
show_metrics(test_predictions, y_test)

=== МЕТРИКИ НА ТРЕНИРОВОЧНЫХ ДАННЫХ ===
MEAN:  $181,313
MAE:  $15,750
RELATIVE MAE ERROR:  0.087
RMSE: $28,576
RELATIVE RMSE ERROR:  0.158
R²:   0.7823
----------------------------------
=== МЕТРИКИ НА ТЕСТОВЫХ ДАННЫХ ===
MEAN:  $180,008
MAE:  $19,316
RELATIVE MAE ERROR:  0.107
RMSE: $34,689
RELATIVE RMSE ERROR:  0.193
R²:   0.6807


### Подбор гиперпараметров для RandomForest

In [27]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from scipy.stats import randint, uniform, loguniform
import joblib

Гиперпараметры для RandomForest

In [ ]:
rf_model = RandomForestRegressor(random_state=42)

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),    
    ('model', rf_model)
])

rf_param_grid = {
    'model': rf_model,
    'model__n_estimators': [50, 100, 200],        
    'model__max_depth': [10, 20, None],           
    'model__min_samples_split': [2, 5],           
    'model__min_samples_leaf': [1, 2],            
    'pca__n_components': [0.7, 0.8, None]        
}

rf_param_distribution = {    
    'model__n_estimators': randint(100, 400),   
    'model__max_depth': randint(10, 20),   
    'model__min_samples_split': randint(5, 20),
    'model__min_samples_leaf': randint(2, 10),
    'model__max_features': [0.5, 0.7, 0.9],
    'model__bootstrap': [True, False]
}

grid_search_rf = GridSearchCV(
                       estimator=rf_pipeline,
                       param_grid=rf_param_grid,
                       cv=5,
                       scoring='neg_mean_absolute_error',
                       n_jobs=-1,
                       verbose=3)

random_search_rf = RandomizedSearchCV(
                        estimator=rf_pipeline,
                        param_distributions=rf_param_distribution,
                        n_iter=100,
                        cv=5,
                        scoring='neg_mean_absolute_error',
                        n_jobs=-1,                      
                        verbose=2,                      
                        random_state=42)

# grid_search_rf.fit(X_train, y_train)

if (config.model_storage_path / 'rf_best_model.pkl').exists():
    rf_best_model = joblib.load(config.model_storage_path / 'rf_best_model.pkl')
    print(f"Загружена модель из {config.model_storage_path}")
else:   
    random_search_rf.fit(X_train, y_train)
    rf_best_model = random_search_rf.best_estimator_
    joblib.dump(rf_best_model, config.model_storage_path / 'rf_best_model.pkl')
    print(f"Модель обучена и сохранена в {config.model_storage_path}")

Загружена модель из models


In [ ]:
print('Лучшие параметры RandomForest:')
print(rf_best_model.get_params()['model'])

Лучшие параметры LightGBM:
RandomForestRegressor(max_depth=11, max_features=0.5, min_samples_leaf=4,
                      min_samples_split=6, n_estimators=134, random_state=42)
<class 'sklearn.ensemble._forest.RandomForestRegressor'>


In [37]:
train_preds = rf_best_model.predict(X_train)
test_preds = rf_best_model.predict(X_test)

print("=== МЕТРИКИ ЛУЧШЕЙ МОДЕЛИ НА ТРЕНИРОВОЧНЫХ ДАННЫХ ===")
show_metrics(train_preds, y_train)

print("----------------------------------")

print("=== МЕТРИКИ ЛУЧШЕЙ МОДЕЛИ НА ТЕСТОВЫХ ДАННЫХ ===")
show_metrics(test_preds, y_test)

=== МЕТРИКИ ЛУЧШЕЙ МОДЕЛИ НА ТРЕНИРОВОЧНЫХ ДАННЫХ ===
MEAN:  $181,313
MAE:  $10,754
RELATIVE MAE ERROR:  0.059
RMSE: $20,157
RELATIVE RMSE ERROR:  0.111
R²:   0.9154
----------------------------------
=== МЕТРИКИ ЛУЧШЕЙ МОДЕЛИ НА ТЕСТОВЫХ ДАННЫХ ===
MEAN:  $180,008
MAE:  $16,521
RELATIVE MAE ERROR:  0.092
RMSE: $27,482
RELATIVE RMSE ERROR:  0.153
R²:   0.8476


c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [10, 14, 15, 29] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


### Подбор гиперпараметров для LightGBL

In [38]:
import lightgbm as lgb

lgb_model = lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)

lgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', lgb_model)
])

param_dist_lgb = {
    'model__n_estimators': randint(100, 500),
    'model__learning_rate': uniform(0.01, 0.2),
    'model__num_leaves': randint(20, 80),
    'model__max_depth': randint(4, 20),
    'model__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'model__colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'model__min_child_samples': randint(5, 50),
    'model__reg_alpha': loguniform(1e-5, 1e2),
    'model__reg_lambda': loguniform(1e-5, 1e2)
}

random_search_lgb = RandomizedSearchCV(
    estimator=lgb_pipeline,
    param_distributions=param_dist_lgb,
    n_iter=100,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=2,
    random_state=42
)

if (config.model_storage_path / 'lgb_best_model.pkl').exists():
    lgb_best_model = joblib.load(config.model_storage_path / 'lgb_best_model.pkl')
    print(f"Загружена модель из {config.model_storage_path}")
else:   
    random_search_lgb.fit(X_train, y_train)
    lgb_best_model = random_search_lgb.best_estimator_
    joblib.dump(lgb_best_model, config.model_storage_path / 'lgb_best_model.pkl')
    print(f"Модель обучена и сохранена в {config.model_storage_path}")

Загружена модель из models


In [39]:
print('Лучшие параметры LightGBM:')
print(lgb_best_model.get_params()['model'])

Лучшие параметры LightGBM:
LGBMRegressor(colsample_bytree=0.6,
              learning_rate=np.float64(0.013615072723104174), max_depth=17,
              min_child_samples=25, n_estimators=403, n_jobs=-1, num_leaves=39,
              random_state=42, reg_alpha=np.float64(0.0036752043516458765),
              reg_lambda=np.float64(1.6188017357746958), subsample=0.8,
              verbose=-1)


In [42]:
train_preds_lgb = lgb_best_model.predict(X_train)
test_preds_lgb = lgb_best_model.predict(X_test)

print('=== МЕТРИКИ ЛУЧШЕЙ LightGBM МОДЕЛИ НА ТРЕНИРОВОЧНЫХ ДАННЫХ ===')
show_metrics(train_preds_lgb, y_train)
print('----------------------------------')
print('=== МЕТРИКИ ЛУЧШЕЙ LightGBM МОДЕЛИ НА ТЕСТОВЫХ ДАННЫХ ===')
show_metrics(test_preds_lgb, y_test)

=== МЕТРИКИ ЛУЧШЕЙ LightGBM МОДЕЛИ НА ТРЕНИРОВОЧНЫХ ДАННЫХ ===
MEAN:  $181,313
MAE:  $8,807
RELATIVE MAE ERROR:  0.049
RMSE: $19,170
RELATIVE RMSE ERROR:  0.106
R²:   0.9299
----------------------------------
=== МЕТРИКИ ЛУЧШЕЙ LightGBM МОДЕЛИ НА ТЕСТОВЫХ ДАННЫХ ===
MEAN:  $180,008
MAE:  $15,766
RELATIVE MAE ERROR:  0.088
RMSE: $27,030
RELATIVE RMSE ERROR:  0.15
R²:   0.8680


c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [10, 14, 15, 29] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


### Подбор гиперпараметров для XGBoost

In [44]:
import xgboost as xgb

xgb_model = xgb.XGBRegressor(random_state=42, n_jobs=-1)

xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb_model)
])

param_dist_xgb = {
    'model__n_estimators': randint(100, 500),
    'model__learning_rate': uniform(0.01, 0.2),
    'model__max_depth': randint(3, 10),
    'model__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'model__colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'model__reg_alpha': loguniform(1e-5, 1e2),
    'model__reg_lambda': loguniform(1e-5, 1e2)
}

random_search_xgb = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_dist_xgb,
    n_iter=100,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=2,
    random_state=42
)

if (config.model_storage_path / 'xgb_best_model.pkl').exists():
    xgb_best_model = joblib.load(config.model_storage_path / 'xgb_best_model.pkl')
    print(f"Загружена модель из {config.model_storage_path}")
else:   
    random_search_xgb.fit(X_train, y_train)
    xgb_best_model = random_search_xgb.best_estimator_
    joblib.dump(xgb_best_model, config.model_storage_path / 'xgb_best_model.pkl')
    print(f"Модель обучена и сохранена в {config.model_storage_path}")

Загружена модель из models


In [45]:
print('Лучшие параметры XGBoost:')
print(xgb_best_model.get_params()['model'])

Лучшие параметры XGBoost:
XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.6, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=np.float64(0.11105047448957144), max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=3, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=441, n_jobs=-1,
             num_parallel_tree=None, ...)


In [ ]:
train_preds_xgb = xgb_best_model.predict(X_train)
test_preds_xgb = xgb_best_model.predict(X_test)

print('=== МЕТРИКИ ЛУЧШЕЙ XGBoost МОДЕЛИ НА ТРЕНИРОВОЧНЫХ ДАННЫХ ===')
show_metrics(train_preds_xgb, y_train)
print('----------------------------------')
print('=== МЕТРИКИ ЛУЧШЕЙ XGBoost МОДЕЛИ НА ТЕСТОВЫХ ДАННЫХ ===')
show_metrics(test_preds_xgb, y_test)


=== МЕТРИКИ ЛУЧШЕЙ XGBoost МОДЕЛИ НА ТРЕНИРОВОЧНЫХ ДАННЫХ ===
MEAN:  $181,313
MAE:  $4,069
RELATIVE MAE ERROR:  0.022
RMSE: $5,199
RELATIVE RMSE ERROR:  0.029
R²:   0.9954
----------------------------------
=== МЕТРИКИ ЛУЧШЕЙ XGBoost МОДЕЛИ НА ТЕСТОВЫХ ДАННЫХ ===
MEAN:  $180,008
MAE:  $15,778
RELATIVE MAE ERROR:  0.088
RMSE: $24,528
RELATIVE RMSE ERROR:  0.136
R²:   0.8972


c:\Users\matve\Desktop\ames-housing\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [10, 14, 15, 29] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
